In [1]:
import os
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np

In [2]:
os.chdir("/home/nlp/achimoa/workspace/hebrew_text_retrieval")

In [3]:
model_name_or_path = "/home/nlp/achimoa/workspace/HebrewModernBERT/outputs/hf/HebrewModernBERT_base_mixed_h50e25c25_1024_0.1"

In [4]:
model = AutoModel.from_pretrained(model_name_or_path)
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

In [14]:
# pip install transformers datasets numpy
from transformers import AutoTokenizer
import numpy as np

def show_tokenization(text, max_length=128, verbose=True):
    print("\n=== TEXT ===")
    print(text)
    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        return_tensors=None,
        return_offsets_mapping=True,  # needs fast tokenizer
    )
    ids   = enc["input_ids"]
    atn   = enc["attention_mask"]
    offs  = enc["offset_mapping"]
    tokens = tokenizer.convert_ids_to_tokens(ids)

    unk_id = tokenizer.unk_token_id
    unk_mask = [t_id == unk_id for t_id in ids]
    unk_ratio = sum(unk_mask) / max(1, len(ids))

    print(f"\nTokenizer: {model_name_or_path}")
    print(f"Fast: {getattr(tokenizer, 'is_fast', False)} | vocab size: {tokenizer.vocab_size} | unk_token: {tokenizer.unk_token}")
    # Normalizer & pre-tokenizer (if available)
    try:
        print("Normalizer:", tokenizer.backend_tokenizer.normalizer)
        print("PreTokenizer:", tokenizer.backend_tokenizer.pre_tokenizer)
    except Exception:
        pass

    print("\nidx | token              | id       | [unk] | offset(start,end)")
    print("-"*70)
    for i, (t, t_id, is_unk, (a,b)) in enumerate(zip(tokens, ids, unk_mask, offs)):
        print(f"{i:>3} | {repr(t):<18} | {t_id:<8} | {str(is_unk):<5} | ({a},{b})")

    if verbose:
        print(f"\n[STATS] total tokens: {len(ids)}, attention on: {sum(atn)}")
        print(f"[STATS] UNK tokens: {sum(unk_mask)} ({unk_ratio:.3%})")

    return {"ids": ids, "tokens": tokens, "unk_mask": unk_mask, "offsets": offs}

# --- Try a few Hebrew sentences (reuse your earlier examples) ---
samples = [
    "אפל הכריזה על אייפון חדש.",
    "הושקה גרסה חדשה של האייפון.",
    "הרכבת לירושלים מאחרת.",
    "הטיסה לא תצא בשל סופה חזקה.",
    "המסעדה פתוחה עד חצות.",
]

for s in samples:
    show_tokenization(s)



=== TEXT ===
אפל הכריזה על אייפון חדש.

Tokenizer: /home/nlp/achimoa/workspace/HebrewModernBERT/outputs/hf/HebrewModernBERT_base_mixed_h50e25c25_1024_0.1
Fast: True | vocab size: 100000 | unk_token: [UNK]
Normalizer: Sequence(normalizers=[Replace(pattern=String("``"), content="""), Replace(pattern=String("''"), content="""), NFKD(), StripAccents(), Lowercase(), ...])
PreTokenizer: Metaspace(replacement="▁", prepend_scheme=always, split=True)

idx | token              | id       | [unk] | offset(start,end)
----------------------------------------------------------------------
  0 | '[CLS]'            | 2        | False | (0,0)
  1 | '▁אפל'             | 3115     | False | (0,3)
  2 | '▁הכריזה'          | 13593    | False | (3,10)
  3 | '▁על'              | 86       | False | (10,13)
  4 | '▁אייפון'          | 13364    | False | (13,20)
  5 | '▁חדש'             | 1436     | False | (20,24)
  6 | '.'                | 99893    | False | (24,25)
  7 | '[SEP]'            | 3        | False 

In [5]:
text = "שרת הרכש הציבורי"
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)
print(outputs.last_hidden_state)

tensor([[[ 0.0629,  0.1785,  0.0771,  ..., -0.0087,  0.0311,  0.1462],
         [-0.7077, -0.2574,  0.7424,  ...,  0.0629, -0.6103, -1.3958],
         [ 0.2553,  0.2532, -0.8703,  ..., -0.3022,  0.4875, -1.5380],
         [ 0.2369,  0.1402, -0.2309,  ...,  0.4035, -0.8641, -2.0621],
         [-0.0084, -0.0351, -0.0459,  ..., -0.1061, -0.0490, -0.0141]]],
       grad_fn=<NativeLayerNormBackward0>)


In [6]:
# --- pip: transformers torch scipy scikit-learn numpy ---
import torch, numpy as np
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from scipy.stats import spearmanr

MODEL   = model_name_or_path  # <-- change to your model
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 512
POOLING = "cls"   # options: "cls" or "mean"

tok = AutoTokenizer.from_pretrained(MODEL, use_fast=True)
enc = AutoModel.from_pretrained(MODEL).to(DEVICE).eval()

def pool(last_hidden_state, attention_mask, method="mean"):
    if method == "cls":
        # index 0 is [CLS] (or <s>), works for BERT/Roberta/ModernBERT-like encoders
        return last_hidden_state[:, 0]
    elif method == "mean":
        mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
        summed = (last_hidden_state * mask).sum(1)
        counts = mask.sum(1).clamp(min=1e-6)
        return summed / counts
    else:
        raise ValueError(f"Unknown pooling: {method}")

@torch.no_grad()
def embed(texts, batch_size=32):
    outs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        x = tok(batch, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        y = enc(**x).last_hidden_state
        s = pool(y, x["attention_mask"], POOLING)
        s = F.normalize(s, p=2, dim=1)  # L2 -> cosine by dot
        outs.append(s.cpu())
    return torch.cat(outs, 0).numpy()

# ---------- tiny labeled hebrew pairs (1=similar, 0=not) ----------
pairs = [
    # similar
    ("אפל הכריזה על אייפון חדש.", "הושקה גרסה חדשה של האייפון.", 1),
    ("הטמפרטורה תרד הלילה.", "מחר יהיה קר מהרגיל.", 1),
    ("הרכבת לירושלים מאחרת.", "הקו המהיר לירושלים נעצר זמנית.", 1),
    ("השראה ליזמות מגיעה מהשטח.", "הרעיונות הכי טובים צומחים מהניסיון.", 1),
    ("הטיסה בוטלה בגלל מזג האוויר.", "הטיסה לא תצא בשל סופה חזקה.", 1),
    ("הקבוצה ניצחה 1:0 במשחק חוץ.", "הקבוצה השיגה ניצחון דחוק מחוץ לבית.", 1),
    # not similar
    ("המסעדה פתוחה עד חצות.", "הילד פתר את המשוואה.", 0),
    ("החלפתי צמיג בפנצ'ריה.", "הספר יצא לאור בשנה הבאה.", 0),
    ("היום חם מאוד בדרום.", "הופעת הג'אז נדחתה לשבוע הבא.", 0),
    ("התחלתי קורס בפייתון.", "הכינרת עלתה במפלס החורף.", 0),
    ("השוק עלה באחוז.", "האוטובוס לרחובות יוצא כל רבע שעה.", 0),
    ("הזמנתי מקום למסעדה טבעונית.", "הקבוצה נשרה מהליגה הארצית.", 0),
]
s1 = [a for a,b,y in pairs]
s2 = [b for a,b,y in pairs]
y  = np.array([y for a,b,y in pairs], dtype=int)

E1 = embed(s1)
E2 = embed(s2)
cos = (E1 * E2).sum(1)  # cosine similarity in [-1,1]

# --- basic stats ---
pos = cos[y==1]; neg = cos[y==0]
print("Pooling:", POOLING)
print("N pos/neg:", len(pos), len(neg))
print(f"mean±std  pos: {pos.mean():.3f}±{pos.std():.3f} | neg: {neg.mean():.3f}±{neg.std():.3f}")
print(f"gap (pos-neg): {pos.mean()-neg.mean():.3f}")

# --- AUC ---
try:
    auc = roc_auc_score(y, cos)
    print(f"AUC: {auc:.3f}")
except Exception as e:
    print("AUC error:", e)

# --- best-F1 threshold search ---
ts = np.linspace(-1.0, 1.0, 401)
f1s = [f1_score(y, (cos >= t).astype(int)) for t in ts]
t_best = ts[int(np.argmax(f1s))]
print(f"Best F1: {max(f1s):.3f} @ threshold {t_best:.2f}")

# Confusion at best threshold
yh = (cos >= t_best).astype(int)
cm = confusion_matrix(y, yh, labels=[1,0])
print("Confusion [ [TP, FN], [FP, TN] ]:\n", cm)

# --- tiny retrieval sanity check (Recall@1, MRR) ---
queries = [
    "אפל מציגה דגם חדש של האייפון.",
    "הלילה יהיה קר במיוחד.",
    "הנסיעה המהירה לירושלים הופסקה.",
    "הרעיונות העסקיים הטובים נוצרים מתוך עשייה.",
    "טיסות בוטלו בשל סערה.",
    "הקבוצה ניצחה בחוץ בתוצאה נמוכה.",
]
docs = [
    "הושקה גרסה חדשה של האייפון.",
    "מחר יהיה קר מהרגיל.",
    "הקו המהיר לירושלים נעצר זמנית.",
    "הרעיונות הכי טובים צומחים מהניסיון.",
    "הטיסה לא תצא בשל סופה חזקה.",
    "הקבוצה השיגה ניצחון דחוק מחוץ לבית.",
    "המסעדה פתוחה עד חצות.",
    "פתרתי את התרגיל במתמטיקה.",
    "היום חם מאוד בדרום.",
    "הזמנתי מקום למסעדה טבעונית.",
]

Q = embed(queries)
D = embed(docs)
S = Q @ D.T  # [nq, nd]

def recall_at_1_and_mrr(S):
    nq = S.shape[0]
    pos_idx = np.arange(6)
    ranks = []
    hits = 0
    for i in range(nq):
        order = np.argsort(-S[i])
        r = int(np.where(order == pos_idx[i])[0][0]) + 1
        ranks.append(r)
        hits += (r == 1)
    recall1 = hits / nq
    mrr = np.mean([1.0/r for r in ranks])
    return recall1, mrr, ranks

rec1, mrr, ranks = recall_at_1_and_mrr(S)
print(f"Retrieval: Recall@1={rec1:.3f}, MRR={mrr:.3f}, ranks={ranks}")

rho, p = spearmanr(cos, y)
print(f"Spearman(cos, label): {rho:.3f} (p={p:.3g})")

Pooling: cls
N pos/neg: 6 6
mean±std  pos: 1.000±0.000 | neg: 1.000±0.000
gap (pos-neg): 0.000
AUC: 0.889
Best F1: 0.667 @ threshold -1.00
Confusion [ [TP, FN], [FP, TN] ]:
 [[6 0]
 [6 0]]
Retrieval: Recall@1=1.000, MRR=1.000, ranks=[1, 1, 1, 1, 1, 1]
Spearman(cos, label): 0.676 (p=0.0158)


In [12]:
# --- pip: transformers torch scipy scikit-learn numpy ---
import torch, numpy as np
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from sklearn.decomposition import PCA
from scipy.stats import spearmanr

MODEL   = model_name_or_path 
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 512
POOLING = "cls"   # options: "cls" or "mean"

# ---- anisotropy mitigation switches ----
APPLY_PC_REMOVAL = True   # turn on/off PC removal
PC_K = 2                  # remove top-k principal components (try 1..3)

tok = AutoTokenizer.from_pretrained(MODEL, use_fast=True)
enc = AutoModel.from_pretrained(MODEL).to(DEVICE).eval()

def pool(last_hidden_state, attention_mask, method="mean"):
    if method == "cls":
        return last_hidden_state[:, 0]
    elif method == "mean":
        mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
        summed = (last_hidden_state * mask).sum(1)
        counts = mask.sum(1).clamp(min=1e-6)
        return summed / counts
    else:
        raise ValueError(f"Unknown pooling: {method}")

@torch.no_grad()
def embed_base(texts, batch_size=32):
    """Raw sentence embeddings (pooled + L2), no PC removal."""
    outs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        x = tok(batch, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        y = enc(**x).last_hidden_state
        s = pool(y, x["attention_mask"], POOLING)
        s = F.normalize(s, p=2, dim=1)  # L2 -> cosine by dot
        outs.append(s.cpu())
    return torch.cat(outs, 0).numpy()

# ---------- tiny labeled hebrew pairs (1=similar, 0=not) ----------
pairs = [
    # similar
    ("אפל הכריזה על אייפון חדש.", "הושקה גרסה חדשה של האייפון.", 1),
    ("הטמפרטורה תרד הלילה.", "מחר יהיה קר מהרגיל.", 1),
    ("הרכבת לירושלים מאחרת.", "הקו המהיר לירושלים נעצר זמנית.", 1),
    ("השראה ליזמות מגיעה מהשטח.", "הרעיונות הכי טובים צומחים מהניסיון.", 1),
    ("הטיסה בוטלה בגלל מזג האוויר.", "הטיסה לא תצא בשל סופה חזקה.", 1),
    ("הקבוצה ניצחה 1:0 במשחק חוץ.", "הקבוצה השיגה ניצחון דחוק מחוץ לבית.", 1),
    # not similar
    ("המסעדה פתוחה עד חצות.", "הילד פתר את המשוואה.", 0),
    ("החלפתי צמיג בפנצ'ריה.", "הספר יצא לאור בשנה הבאה.", 0),
    ("היום חם מאוד בדרום.", "הופעת הג'אז נדחתה לשבוע הבא.", 0),
    ("התחלתי קורס בפייתון.", "הכינרת עלתה במפלס החורף.", 0),
    ("השוק עלה באחוז.", "האוטובוס לרחובות יוצא כל רבע שעה.", 0),
    ("הזמנתי מקום למסעדה טבעונית.", "הקבוצה נשרה מהליגה הארצית.", 0),
]
s1 = [a for a,b,y in pairs]
s2 = [b for a,b,y in pairs]
y  = np.array([y for a,b,y in pairs], dtype=int)

# --- retrieval sanity check sets ---
queries = [
    "אפל מציגה דגם חדש של האייפון.",
    "הלילה יהיה קר במיוחד.",
    "הנסיעה המהירה לירושלים הופסקה.",
    "הרעיונות העסקיים הטובים נוצרים מתוך עשייה.",
    "טיסות בוטלו בשל סערה.",
    "הקבוצה ניצחה בחוץ בתוצאה נמוכה.",
]
docs = [
    "הושקה גרסה חדשה של האייפון.",
    "מחר יהיה קר מהרגיל.",
    "הקו המהיר לירושלים נעצר זמנית.",
    "הרעיונות הכי טובים צומחים מהניסיון.",
    "הטיסה לא תצא בשל סופה חזקה.",
    "הקבוצה השיגה ניצחון דחוק מחוץ לבית.",
    "המסעדה פתוחה עד חצות.",
    "פתרתי את התרגיל במתמטיקה.",
    "היום חם מאוד בדרום.",
    "הזמנתי מקום למסעדה טבעונית.",
]

# ---- Fit PCs on reference embeddings (same pooling) ----
# Use a broader reference if you have one; this is fine to start.
def fit_pc_removal(E_ref, k=2):
    pca = PCA(n_components=k, svd_solver="full").fit(E_ref)
    U = pca.components_  # [k, d], orthonormal
    return U, pca.explained_variance_ratio_

def remove_pc(E, U):
    proj = (E @ U.T) @ U      # project onto span(U)
    E_ = E - proj
    E_ /= np.linalg.norm(E_, axis=1, keepdims=True).clip(1e-12, None)
    return E_

U = None
if APPLY_PC_REMOVAL:
    ref_texts = s1 + s2 + queries + docs
    E_ref = embed_base(ref_texts)
    U, evr = fit_pc_removal(E_ref, k=PC_K)
    print(f"PC removal: ON (k={PC_K}) | EVR removed = {np.round(evr, 4)}")
else:
    print("PC removal: OFF")

def embed(texts, batch_size=32):
    E = embed_base(texts, batch_size)
    if APPLY_PC_REMOVAL and U is not None:
        E = remove_pc(E, U)
    return E

# --------- Evaluate pairs ----------
E1 = embed(s1)
E2 = embed(s2)
cos = (E1 * E2).sum(1)  # cosine similarity in [-1,1]

# --- basic stats ---
pos = cos[y==1]; neg = cos[y==0]
print("Pooling:", POOLING)
print("N pos/neg:", len(pos), len(neg))
print(f"mean±std  pos: {pos.mean():.3f}±{pos.std():.3f} | neg: {neg.mean():.3f}±{neg.std():.3f}")
print(f"gap (pos-neg): {pos.mean()-neg.mean():.3f}")

# --- AUC ---
try:
    auc = roc_auc_score(y, cos)
    print(f"AUC: {auc:.3f}")
except Exception as e:
    print("AUC error:", e)

# --- best-F1 threshold search ---
ts = np.linspace(-1.0, 1.0, 401)
f1s = [f1_score(y, (cos >= t).astype(int)) for t in ts]
t_best = ts[int(np.argmax(f1s))]
print(f"Best F1: {max(f1s):.3f} @ threshold {t_best:.2f}")

# Confusion at best threshold
yh = (cos >= t_best).astype(int)
cm = confusion_matrix(y, yh, labels=[1,0])
print("Confusion [ [TP, FN], [FP, TN] ]:\n", cm)

# --- tiny retrieval sanity check (Recall@1, MRR) ---
Q = embed(queries)
D = embed(docs)
S = Q @ D.T  # [nq, nd]

def recall_at_1_and_mrr(S):
    nq = S.shape[0]
    pos_idx = np.arange(6)
    ranks = []
    hits = 0
    for i in range(nq):
        order = np.argsort(-S[i])
        r = int(np.where(order == pos_idx[i])[0][0]) + 1
        ranks.append(r)
        hits += (r == 1)
    recall1 = hits / nq
    mrr = np.mean([1.0/r for r in ranks])
    return recall1, mrr, ranks

rec1, mrr, ranks = recall_at_1_and_mrr(S)
print(f"Retrieval: Recall@1={rec1:.3f}, MRR={mrr:.3f}, ranks={ranks}")

rho, p = spearmanr(cos, y)
print(f"Spearman(cos, label): {rho:.3f} (p={p:.3g})")


PC removal: ON (k=2) | EVR removed = [0.3601 0.1618]
Pooling: cls
N pos/neg: 6 6
mean±std  pos: 1.000±0.000 | neg: 1.000±0.000
gap (pos-neg): 0.000
AUC: 0.944
Best F1: 0.667 @ threshold -1.00
Confusion [ [TP, FN], [FP, TN] ]:
 [[6 0]
 [6 0]]
Retrieval: Recall@1=0.667, MRR=0.833, ranks=[1, 2, 1, 1, 2, 1]
Spearman(cos, label): 0.772 (p=0.00323)


In [13]:
# --- pip: transformers torch scipy scikit-learn numpy ---
import torch, numpy as np
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from sklearn.decomposition import PCA
from scipy.stats import spearmanr

MODEL   = "dicta-il/dictabert"
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 512
POOLING = "cls"   # options: "cls" or "mean"

# ---- anisotropy mitigation switches ----
APPLY_PC_REMOVAL = True   # turn on/off PC removal
PC_K = 2                  # remove top-k principal components (try 1..3)

tok = AutoTokenizer.from_pretrained(MODEL, use_fast=True)
enc = AutoModel.from_pretrained(MODEL).to(DEVICE).eval()

def pool(last_hidden_state, attention_mask, method="mean"):
    if method == "cls":
        return last_hidden_state[:, 0]
    elif method == "mean":
        mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
        summed = (last_hidden_state * mask).sum(1)
        counts = mask.sum(1).clamp(min=1e-6)
        return summed / counts
    else:
        raise ValueError(f"Unknown pooling: {method}")

@torch.no_grad()
def embed_base(texts, batch_size=32):
    """Raw sentence embeddings (pooled + L2), no PC removal."""
    outs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        x = tok(batch, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        y = enc(**x).last_hidden_state
        s = pool(y, x["attention_mask"], POOLING)
        s = F.normalize(s, p=2, dim=1)  # L2 -> cosine by dot
        outs.append(s.cpu())
    return torch.cat(outs, 0).numpy()

# ---------- tiny labeled hebrew pairs (1=similar, 0=not) ----------
pairs = [
    # similar
    ("אפל הכריזה על אייפון חדש.", "הושקה גרסה חדשה של האייפון.", 1),
    ("הטמפרטורה תרד הלילה.", "מחר יהיה קר מהרגיל.", 1),
    ("הרכבת לירושלים מאחרת.", "הקו המהיר לירושלים נעצר זמנית.", 1),
    ("השראה ליזמות מגיעה מהשטח.", "הרעיונות הכי טובים צומחים מהניסיון.", 1),
    ("הטיסה בוטלה בגלל מזג האוויר.", "הטיסה לא תצא בשל סופה חזקה.", 1),
    ("הקבוצה ניצחה 1:0 במשחק חוץ.", "הקבוצה השיגה ניצחון דחוק מחוץ לבית.", 1),
    # not similar
    ("המסעדה פתוחה עד חצות.", "הילד פתר את המשוואה.", 0),
    ("החלפתי צמיג בפנצ'ריה.", "הספר יצא לאור בשנה הבאה.", 0),
    ("היום חם מאוד בדרום.", "הופעת הג'אז נדחתה לשבוע הבא.", 0),
    ("התחלתי קורס בפייתון.", "הכינרת עלתה במפלס החורף.", 0),
    ("השוק עלה באחוז.", "האוטובוס לרחובות יוצא כל רבע שעה.", 0),
    ("הזמנתי מקום למסעדה טבעונית.", "הקבוצה נשרה מהליגה הארצית.", 0),
]
s1 = [a for a,b,y in pairs]
s2 = [b for a,b,y in pairs]
y  = np.array([y for a,b,y in pairs], dtype=int)

# --- retrieval sanity check sets ---
queries = [
    "אפל מציגה דגם חדש של האייפון.",
    "הלילה יהיה קר במיוחד.",
    "הנסיעה המהירה לירושלים הופסקה.",
    "הרעיונות העסקיים הטובים נוצרים מתוך עשייה.",
    "טיסות בוטלו בשל סערה.",
    "הקבוצה ניצחה בחוץ בתוצאה נמוכה.",
]
docs = [
    "הושקה גרסה חדשה של האייפון.",
    "מחר יהיה קר מהרגיל.",
    "הקו המהיר לירושלים נעצר זמנית.",
    "הרעיונות הכי טובים צומחים מהניסיון.",
    "הטיסה לא תצא בשל סופה חזקה.",
    "הקבוצה השיגה ניצחון דחוק מחוץ לבית.",
    "המסעדה פתוחה עד חצות.",
    "פתרתי את התרגיל במתמטיקה.",
    "היום חם מאוד בדרום.",
    "הזמנתי מקום למסעדה טבעונית.",
]

# ---- Fit PCs on reference embeddings (same pooling) ----
# Use a broader reference if you have one; this is fine to start.
def fit_pc_removal(E_ref, k=2):
    pca = PCA(n_components=k, svd_solver="full").fit(E_ref)
    U = pca.components_  # [k, d], orthonormal
    return U, pca.explained_variance_ratio_

def remove_pc(E, U):
    proj = (E @ U.T) @ U      # project onto span(U)
    E_ = E - proj
    E_ /= np.linalg.norm(E_, axis=1, keepdims=True).clip(1e-12, None)
    return E_

U = None
if APPLY_PC_REMOVAL:
    ref_texts = s1 + s2 + queries + docs
    E_ref = embed_base(ref_texts)
    U, evr = fit_pc_removal(E_ref, k=PC_K)
    print(f"PC removal: ON (k={PC_K}) | EVR removed = {np.round(evr, 4)}")
else:
    print("PC removal: OFF")

def embed(texts, batch_size=32):
    E = embed_base(texts, batch_size)
    if APPLY_PC_REMOVAL and U is not None:
        E = remove_pc(E, U)
    return E

# --------- Evaluate pairs ----------
E1 = embed(s1)
E2 = embed(s2)
cos = (E1 * E2).sum(1)  # cosine similarity in [-1,1]

# --- basic stats ---
pos = cos[y==1]; neg = cos[y==0]
print("Pooling:", POOLING)
print("N pos/neg:", len(pos), len(neg))
print(f"mean±std  pos: {pos.mean():.3f}±{pos.std():.3f} | neg: {neg.mean():.3f}±{neg.std():.3f}")
print(f"gap (pos-neg): {pos.mean()-neg.mean():.3f}")

# --- AUC ---
try:
    auc = roc_auc_score(y, cos)
    print(f"AUC: {auc:.3f}")
except Exception as e:
    print("AUC error:", e)

# --- best-F1 threshold search ---
ts = np.linspace(-1.0, 1.0, 401)
f1s = [f1_score(y, (cos >= t).astype(int)) for t in ts]
t_best = ts[int(np.argmax(f1s))]
print(f"Best F1: {max(f1s):.3f} @ threshold {t_best:.2f}")

# Confusion at best threshold
yh = (cos >= t_best).astype(int)
cm = confusion_matrix(y, yh, labels=[1,0])
print("Confusion [ [TP, FN], [FP, TN] ]:\n", cm)

# --- tiny retrieval sanity check (Recall@1, MRR) ---
Q = embed(queries)
D = embed(docs)
S = Q @ D.T  # [nq, nd]

def recall_at_1_and_mrr(S):
    nq = S.shape[0]
    pos_idx = np.arange(6)
    ranks = []
    hits = 0
    for i in range(nq):
        order = np.argsort(-S[i])
        r = int(np.where(order == pos_idx[i])[0][0]) + 1
        ranks.append(r)
        hits += (r == 1)
    recall1 = hits / nq
    mrr = np.mean([1.0/r for r in ranks])
    return recall1, mrr, ranks

rec1, mrr, ranks = recall_at_1_and_mrr(S)
print(f"Retrieval: Recall@1={rec1:.3f}, MRR={mrr:.3f}, ranks={ranks}")

rho, p = spearmanr(cos, y)
print(f"Spearman(cos, label): {rho:.3f} (p={p:.3g})")


Some weights of BertModel were not initialized from the model checkpoint at dicta-il/dictabert and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PC removal: ON (k=2) | EVR removed = [0.1798 0.1419]
Pooling: cls
N pos/neg: 6 6
mean±std  pos: 0.949±0.007 | neg: 0.862±0.026
gap (pos-neg): 0.087
AUC: 1.000
Best F1: 1.000 @ threshold 0.90
Confusion [ [TP, FN], [FP, TN] ]:
 [[6 0]
 [0 6]]
Retrieval: Recall@1=1.000, MRR=1.000, ranks=[1, 1, 1, 1, 1, 1]
Spearman(cos, label): 0.869 (p=0.000242)


In [8]:
text2 = "שר הרכש הציבורי"
inputs2 = tokenizer(text2, return_tensors="pt")
outputs2 = model(**inputs2)
print(outputs2.last_hidden_state)

tensor([[[ 6.3470e-02,  1.8156e-01,  7.5804e-02,  ..., -1.3113e-03,
           3.1217e-02,  1.5316e-01],
         [-8.2833e-01,  3.4944e-01,  3.3799e-01,  ...,  5.5338e-01,
          -6.7064e-01,  9.3690e-03],
         [ 2.7118e-01, -1.9592e-01, -8.7875e-01,  ...,  3.3450e-01,
           4.7014e-01, -1.4724e+00],
         [ 5.3364e-01, -1.0545e-01, -3.2305e-01,  ...,  5.0104e-01,
          -7.5120e-01, -1.9389e+00],
         [-5.1613e-03, -4.4259e-02, -5.5469e-02,  ..., -9.1513e-02,
          -4.8830e-02, -1.7307e-03]]], grad_fn=<NativeLayerNormBackward0>)


In [ ]:
text = "כדי להמציא יש צורך בדמיון טוב וערימת גרוטאות"

In [6]:
text = "במהלך המערכה הצבאית הממושכת נתקל אלכסנדר הגדול בהתנגדות עיקשת מצד האוכלוסייה המקומית שוחרת העצמאות."